Import Packages

In [1]:
import folium
from folium import PolyLine, CircleMarker
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiLineString, LineString
from shapely.ops import nearest_points
import math
import numpy as np
from mrt_map import get_mrt_map
from bus_route_plot import original_route, new_route, original_and_new_route

Function to get inner threshold values: 
- Calculate the inner threshold for passenger volumes at bus stops located 
between MRT stops for each route, based on a specified quantile.

Parameters:
- all_bus_data (DataFrame): Dataset containing bus stop information for all routes.
-   inner_quantile (float): Quantile value to use for calculating the threshold.
    
Returns:
- float: The threshold value for passenger volumes between MRT stops, or None if no values found.

In [8]:
def create_inner_threshold(all_bus_data, inner_quantile):
    
    in_between_volumes = []  # List to store passenger volumes of in-between stops
    grouped_routes = all_bus_data.groupby(['ServiceNo'])  # Group data by individual bus routes
    processed_stops = set()  # Track stops already processed to avoid duplicates

    # Iterate through each route (by ServiceNo)
    for (serviceno), route in grouped_routes:
        mrt_busstops = route[route['MRTBusStop'] == 1]  # Filter MRT bus stops in the route

        # Proceed only if there are at least 2 MRT stops in the route
        if len(mrt_busstops) >= 2:
            # Identify the sequence positions of the first and last MRT stops
            first_mrt_stop_seq = mrt_busstops['StopSequence'].min()
            last_mrt_stop_seq = mrt_busstops['StopSequence'].max()

            # Select non-MRT stops that fall between the first and last MRT stops
            in_between_stops = route[(route['StopSequence'] > first_mrt_stop_seq) &
                                     (route['StopSequence'] < last_mrt_stop_seq) &
                                     (route['MRTBusStop'] == 0)]

            # Remove any stops that have already been processed to avoid duplicates
            in_between_stops = in_between_stops[~in_between_stops['BusStopCode'].isin(processed_stops)]

            # Add the passenger volume for each distinct in-between stop to the list
            if not in_between_stops.empty:
                for _, stop in in_between_stops.iterrows():
                    if pd.notna(stop['average_passenger_volume']):  # Ensure value is not NaN
                        in_between_volumes.append(stop['average_passenger_volume'])

            # Mark these stops as processed to avoid re-counting
            processed_stops.update(in_between_stops['BusStopCode'])

    # Calculate the quantile value for in-between stop volumes, if any volumes exist
    if in_between_volumes:
        in_between_volumes_series = pd.Series(in_between_volumes)
        inside_threshold_value = in_between_volumes_series.quantile(inner_quantile)
    else:
        inside_threshold_value = None  # Return None if there are no volumes to process

    return inside_threshold_value  # Return the calculated threshold value


Function to get outer threshold values: 
- Calculate the outer threshold for passenger volumes at bus stops along a specific route, based on a specified quantile.
    
Parameters:
- route (DataFrame): Dataset containing bus stop information for an individual route.
- outer_quantile (float): Quantile value to use for calculating the threshold.
    
Returns:
- float: The threshold value for passenger volumes along the route, or None if no values found.

In [9]:
def create_outer_threshold(route, outer_quantile):

    # Extract all passenger volumes from the 'average_passenger_volume' column, excluding NaN values
    all_volumes = route['average_passenger_volume'].dropna().tolist()

    # Calculate the quantile threshold only if there are valid passenger volume values
    if all_volumes:
        all_volumes_series = pd.Series(all_volumes)  # Convert the list to a pandas Series for easy quantile calculation
        outside_threshold_value = all_volumes_series.quantile(outer_quantile)  # Get the specified quantile value
    else:
        outside_threshold_value = None  # Return None if there are no volumes to process

    return outside_threshold_value  # Return the calculated threshold value


Classify bus stops on the top 10 most parallel bus routes as 'keep' or 'remove' based on passenger volume thresholds relative to MRT connectivity.

Parameters:
- all_bus_data (DataFrame): Combined dataset of all bus routes and stops.
- paralleltrunkservicesranked (DataFrame): Ranked data of parallel bus services based on parallelism score.
- inner_quantile (float): Quantile value for determining the inner threshold between MRT stops.
- outer_quantile (float): Quantile value for determining the outer threshold outside MRT stops.

Returns:
- DataFrame: Modified dataset of the top 10 parallel bus routes with an additional 'outcome' column ('keep'/'remove').

In [ ]:
def process_bus_routes(all_bus_data, paralleltrunkservicesranked, inner_quantile, outer_quantile):
    
    # Select the top 10 parallel bus services
    top_10_buses = paralleltrunkservicesranked.head(10)
    top_10_bus_data = all_bus_data[all_bus_data['ServiceNo'].isin(top_10_buses['ServiceNo'])].copy()

    # Group data by bus service number
    grouped_routes = top_10_bus_data.groupby(['ServiceNo'])
    final_routes = []

    # Process each route in the top 10 parallel bus services
    for ServiceNo, route in grouped_routes:
        mrt_busstops = route[route['MRTBusStop'] == 1]
        
        # If route has fewer than 2 MRT stops, keep all stops
        if len(mrt_busstops) < 2:
            route.loc[:, 'outcome'] = 'keep'
        else:
            # Part 1: Process bus stops between MRT stops
            inner_threshold = create_inner_threshold(all_bus_data, inner_quantile)
            mrt_busstops_stop_sequences = mrt_busstops['StopSequence'].tolist()

            # For each pair of consecutive MRT stops
            for i in range(len(mrt_busstops_stop_sequences) - 1):
                first_mrt_busstop_sequence_no = mrt_busstops_stop_sequences[i]
                second_mrt_busstop_sequence_no = mrt_busstops_stop_sequences[i + 1]

                mrt_line_1 = mrt_busstops.iloc[i]['MRTLine'] if pd.notna(mrt_busstops.iloc[i]['MRTLine']) else []
                mrt_line_2 = mrt_busstops.iloc[i + 1]['MRTLine'] if pd.notna(mrt_busstops.iloc[i + 1]['MRTLine']) else []


                # If both MRT stops from same MRT line, evaluate in-between stops
                if set(mrt_line_1) == set(mrt_line_2): 
                    # Get in-between stops and calculate their average passenger volume
                    in_between_stops = route[(route['StopSequence'] > first_mrt_busstop_sequence_no) & 
                                             (route['StopSequence'] < second_mrt_busstop_sequence_no)]
                    avg_in_between_volume = in_between_stops['average_passenger_volume'].mean()

                    # Classify in-between stops based on inner threshold
                    if avg_in_between_volume >= inner_threshold:
                        route.loc[(route['StopSequence'] > first_mrt_busstop_sequence_no) & 
                                  (route['StopSequence'] < second_mrt_busstop_sequence_no), 'outcome'] = 'keep'
                    else:
                        route.loc[(route['StopSequence'] > first_mrt_busstop_sequence_no) & 
                                  (route['StopSequence'] < second_mrt_busstop_sequence_no), 'outcome'] = 'remove'
                # If MRT stops are on different lines, keep all in-between stops
                else:
                    route.loc[(route['StopSequence'] > first_mrt_busstop_sequence_no) & 
                              (route['StopSequence'] < second_mrt_busstop_sequence_no), 'outcome'] = 'keep'

            # Part 2: Process bus stops outside MRT stop boundaries
            outer_threshold = create_outer_threshold(route, outer_quantile)
            first_bus_stop_seq = route['StopSequence'].min()
            last_bus_stop_seq = route['StopSequence'].max()
            first_mrt_busstop_seq = mrt_busstops_stop_sequences[0]
            last_mrt_busstop_seq = mrt_busstops_stop_sequences[-1]

            # Process stops before the first MRT stop
            before_mrt_stops = route[(route['StopSequence'] < first_mrt_busstop_seq) & 
                                     (route['StopSequence'] > first_bus_stop_seq)]
            if not before_mrt_stops.empty:
                for idx, row in before_mrt_stops.iterrows():
                    if row['average_passenger_volume'] < outer_threshold:
                        route.loc[idx, 'outcome'] = 'remove'

            # Process stops after the last MRT stop
            after_mrt_stops = route[(route['StopSequence'] > last_mrt_busstop_seq) & 
                                    (route['StopSequence'] < last_bus_stop_seq)]
            if not after_mrt_stops.empty:
                for idx, row in after_mrt_stops.iterrows():
                    if row['average_passenger_volume'] < outer_threshold:
                        route.loc[idx, 'outcome'] = 'remove'

        # Add the modified route to the final list
        final_routes.append(route)

    # Concatenate all processed routes into a single DataFrame
    final_result = pd.concat(final_routes)

    return final_result


Run the code

In [ ]:
# Load data
all_bus_data = pd.read_csv("Bus_RoutesStopsServices/all_bus_data.csv")
paralleltrunkservicesranked = pd.read_csv("Bus_RoutesStopsServices/paralleltrunkservicesranked.csv")

# Define quartiles
inner_quantile = 0.5
outer_quantile = 0.5

# Run code & save output to CSV
new_top_10_bus_data = process_bus_routes(all_bus_data, paralleltrunkservicesranked, inner_quantile, outer_quantile)
new_top_10_bus_data['ServiceNo'] = new_top_10_bus_data['ServiceNo'].astype(str)
new_top_10_bus_data.to_csv("new_top_10_bus_data.csv", index=False)

# Filter out only the modified routes to put into the interface & output to CSV
top_10_buses_new_routes_only = new_top_10_bus_data.groupby('ServiceNo').filter(lambda x: (x['outcome'] != 'keep').any())
top_10_buses_new_routes_only = top_10_buses_new_routes_only[top_10_buses_new_routes_only["outcome"]=='keep']
top_10_buses_new_routes_only['ServiceNo'] = top_10_buses_new_routes_only['ServiceNo'].astype(str)
top_10_buses_new_routes_only.to_csv("top_10_buses_new_routes_only.csv", index=False)


Plot Routes
- uncomment line by line to view each map accordingly

In [18]:
# # Displays the original route map for bus service(s)
#original_route('16') 

# Displays the modified route map for bus service(s)
#new_route('16')

# Displays both the original and modified routes on the same map for bus service
#original_and_new_route('27')